# Preprocess the INA list of files to extract the Show names and list of issues

In [2]:
import json
import os
import pandas as pd
import numpy as np
from random import randint
from datetime import datetime
from impresso_essentials.utils import ALL_MEDIA, PARTNER_TO_MEDIA
from collections import Counter

In [1]:
DATA_DIR = "../../../data"

# 1. Read the original files and preprocessed files listing all existing information

In [3]:
notice_filepath = f"{DATA_DIR}/sample_data/INA/ListeDesNotices.txt"
files_filepath = f"{DATA_DIR}/sample_data/INA/ListeDesFichiers.txt"

Note - The encodings of these files was not utf-8, we will need to write the new files in utf-8.

**Notices**

In [4]:
notice_df = pd.read_csv(notice_filepath, sep='\t', header=0, encoding = 'latin-1')
notice_df

,Identifiant de la notice,Thèque,Titre collection,Titre propre,Date de diffusion,Date d'enregistrement,Heure de diffusion,Titre phonogramme,Genre,Thématique,Durée
0,PHD85000118,PH (Phono),Le journal sonore de la semaine,Joseph GOEBBELS lit un texte d'Adolf HITLER su...,15/03/1939,15/03/1939,NaN,NaN,Déclaration ;,Politique ;,00:03:46
1,PHD85000121,PH (Phono),Radio journal de France,Léon Blum fait appel aux détenteurs de capitaux,17/07/1936,17/07/1936,NaN,NaN,Journal parlé ;,NaN,00:13:21
2,PHD85000672,PH (Phono),La voix de Paris,Gaston HENRY-HAYE : les projets de la nouvelle...,NaN,20/10/1935,NaN,NaN,Interview entretien ; Journal parlé ;,Tourisme ;,00:02:27
3,PHD85000963,PH (Phono),Radio journal de France,Maurice BOURDET : lecture des informations du ...,07/01/1936,07/01/1936,NaN,NaN,Journal parlé ; Papier ;,Information ;,00:05:57
4,PHD85001640,PH (Phono),Radio journal de France,Appel à la nation de Jules ROMAINS,30/10/1938,30/10/1938,NaN,NaN,Déclaration ; Journal parlé ;,Littérature ; Politique ;,00:25:00
...,...,...,...,...,...,...,...,...,...,...,...
39602,00419281,PH (Phono),Inter actualités de 19H00,Inter soir 19h00 du 29 décembre 1989,29/12/1989,29/12/1989,19:00:00,NaN,Journal parlé ;,NaN,01:01:00
39603,00419356,PH (Phono),Inter actualités de 19H00,Inter soir 19h00 du 30 décembre 1989,30/12/1989,30/12/1989,19:00:00,NaN,Journal parlé ;,NaN,00:55:00
39604,00419416,PH (Phono),Inter actualités de 19H00,Inter soir 19h00 du 31 décembre 1989,31/12/1989,31/12/1989,19:00:00,NaN,Journal parlé ;,NaN,00:15:00
39605,PHD86069295,PH (Phono),NaN,Voyage à Nancy du Maréchal PETAIN,27/05/1944,27/05/1944,NaN,NaN,Journal parlé ; Reportage ;,Information ;,00:05:52


**Files**

In [5]:
files_df = pd.read_csv(files_filepath, sep='\t', header=0, encoding = 'latin-1')
files_df

,Type de notice,Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion,Identifiant notice,Fichier source,Nom du fichier,TC IN,TC OUT,Format,Répertoire,Chemin
0,EMISSION RAD.,Titre: L'Automobile en Europe - Titre collecti...,172961,96F04705SA0005_01.MP3,00172961_96F04705SA0005_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
1,EMISSION RAD.,Titre: Le dimanche des Europeens - Titre colle...,165368,96F04705SA0001_01.MP3,00165368_96F04705SA0001_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
2,EMISSION RAD.,Titre: Enseignement education chomage et emp...,167288,96F04705SA0002_01.MP3,00167288_96F04705SA0002_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
3,EMISSION RAD.,Titre: Le service militaire - Titre collection...,169364,96F04705SA0003_01.MP3,00169364_96F04705SA0003_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
4,EMISSION RAD.,Titre: L'amour en Europe - Titre collection:C'...,171017,96F04705SA0004_01.MP3,00171017_96F04705SA0004_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne
...,...,...,...,...,...,...,...,...,...,...
47725,EMISSION RAD.,Titre: Uri cite musicale barcelone n ?25 - Tit...,PHZ16062539,LD95074_01.MP3,PHZ16062539_LD95074_01___EXPORT.MP3,:::,:::,FVISIO,Université radiophonique internationale,iMPRESSO\Magazines_d'information\Universite_ra...
47726,EMISSION RAD.,Titre: Uri cite musicale barcelone n )26 - Tit...,PHZ16062540,LD95075_01.MP3,PHZ16062540_LD95075_01___EXPORT.MP3,:::,:::,FVISIO,Université radiophonique internationale,iMPRESSO\Magazines_d'information\Universite_ra...
47727,EMISSION RAD.,Titre: L'avenir du protectorat au Maroc - Titr...,PHD86044647,98INA08505PA0144_02.MP3,PHD86044647_98INA08505PA0144_02_530113_1124500...,00:53:01:13,01:12:45:00,FVISIO,08-Tribune_de_Paris_1946-1963,iMPRESSO\Journal_parle\08-Tribune_de_Paris_194...
47728,EMISSION RAD.,Titre: Inter soir 19h00 19H00 du 10 janvier 19...,580978,94F05001SA0010_01.MP3,00580978_94F05001SA0010_01_3090021_3305211_EXP...,03:09:00:21,03:30:52:11,FVISIO,15-_Inter_soir_ou_Inter_actualites_1990-_fin_j...,iMPRESSO\Journal_parle\15-_Inter_soir_ou_Inter...


**Extended Files List**

In [ ]:
ext_files_list_path = f"{DATA_DIR}/data/sample_data/INA/ListeDesFichiers_extended.csv"
ext_files_df = pd.read_csv(ext_files_list_path, index_col=0)
ext_files_df

,Type de notice,Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion,Identifiant notice,Fichier source,Nom du fichier,TC IN,TC OUT,Format,Répertoire,Chemin,Titre propre,Titre collection,Chaine de diffusion,Heure de diffusion,Notice,Date de diffusion
0,EMISSION RAD.,Titre: L'Automobile en Europe - Titre collecti...,172961,96F04705SA0005_01.MP3,00172961_96F04705SA0005_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,L'Automobile en Europe,C'est en France c'est en Europe,Radio France,12:05:00,00172961,06/10/1996
1,EMISSION RAD.,Titre: Le dimanche des Europeens - Titre colle...,165368,96F04705SA0001_01.MP3,00165368_96F04705SA0001_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,Le dimanche des Europeens,C'est en France c'est en Europe,Radio France,12:05:00,00165368,08/09/1996
2,EMISSION RAD.,Titre: Enseignement education chomage et emp...,167288,96F04705SA0002_01.MP3,00167288_96F04705SA0002_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,Enseignement education chomage et emploi,C'est en France c'est en Europe,Radio France,12:05:00,00167288,15/09/1996
3,EMISSION RAD.,Titre: Le service militaire - Titre collection...,169364,96F04705SA0003_01.MP3,00169364_96F04705SA0003_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,Le service militaire,C'est en France c'est en Europe,Radio France,12:05:00,00169364,22/09/1996
4,EMISSION RAD.,Titre: L'amour en Europe - Titre collection:C'...,171017,96F04705SA0004_01.MP3,00171017_96F04705SA0004_01___EXPORT.MP3,:::,:::,FVISIO,Construction européenne,iMPRESSO\Construction_europeenne,L'amour en Europe,C'est en France c'est en Europe,Radio France,12:05:00,00171017,29/09/1996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47725,EMISSION RAD.,Titre: Uri cite musicale barcelone n ?25 - Tit...,PHZ16062539,LD95074_01.MP3,PHZ16062539_LD95074_01___EXPORT.MP3,:::,:::,FVISIO,Université radiophonique internationale,iMPRESSO\Magazines_d'information\Universite_ra...,Uri cite musicale barcelone n ?25,Universite radiophonique internationale,Radio Television Francaise,NaN,PHZ16062539,16/04/964
47726,EMISSION RAD.,Titre: Uri cite musicale barcelone n )26 - Tit...,PHZ16062540,LD95075_01.MP3,PHZ16062540_LD95075_01___EXPORT.MP3,:::,:::,FVISIO,Université radiophonique internationale,iMPRESSO\Magazines_d'information\Universite_ra...,Uri cite musicale barcelone n )26,Universite radiophonique internationale,Radio Television Francaise,NaN,PHZ16062540,16/04/964
47727,EMISSION RAD.,Titre: L'avenir du protectorat au Maroc - Titr...,PHD86044647,98INA08505PA0144_02.MP3,PHD86044647_98INA08505PA0144_02_530113_1124500...,00:53:01:13,01:12:45:00,FVISIO,08-Tribune_de_Paris_1946-1963,iMPRESSO\Journal_parle\08-Tribune_de_Paris_194...,L'avenir du protectorat au Maroc,Tribune de Paris : Les hommes les evenements ...,Radio Television Francaise,NaN,PHD86044647,10/12/1953
47728,EMISSION RAD.,Titre: Inter soir 19h00 19H00 du 10 janvier 19...,580978,94F05001SA0010_01.MP3,00580978_94F05001SA0010_01_3090021_3305211_EXP...,03:09:00:21,03:30:52:11,FVISIO,15-_Inter_soir_ou_Inter_actualites_1990-_fin_j...,iMPRESSO\Journal_parle\15-_Inter_soir_ou_Inter...,Inter soir 19h00 19H00 du 10 janvier 1994,Inter soir 19h00,Radio France,NaN,00580978,10/01/1994


**Collection Aliases**

In [ ]:
aliases_from_files_df_filepath = f"{DATA_DIR}/data/sample_data/INA/collection_aliases_from_files.csv"
aliases_from_files_df = pd.read_csv(aliases_from_files_df_filepath, index_col=0)
aliases_from_files_df

,alias,année_début,année_fin,Titre collection,Nombre de notices,Chaine de diffusion,Répertoire,Identifiants des notices
0,CAPVD,1975,1984,75... 2000 : Comprendre aujourd'hui pour vivre...,107,Radio France,75...2000 : comprendre pour vivre demain,"['PHD95073186', 'PHD95073187', 'PHD95073188', ..."
1,CFCE,1996,1996,C'est en France c'est en Europe,17,Radio France,Construction européenne,"['00172961', '00165368', '00167288', '00169364..."
2,ConsEuro,1949,1960,Conseil de l'Europe,17,Radio Television Francaise,Construction européenne,"['PHD86026156', 'PHD86026894', 'PHD86037808', ..."
3,EcoEH,1962,1968,L'economie et les hommes,295,"['Office Radio Television France', 'Radio Tele...",Les banques suisses,"['PHD90000370', 'PHD94008439', 'PHD94008440', ..."
4,EdSpe,1958,1963,Edition speciale,624,Radio Television Francaise,Edition spéciale,"['PHD88011641', 'PHD88011647', 'PHD88011650', ..."
5,EnjeuxInt,1984,1996,Les enjeux internationaux,1402,Radio France,Les organisations internationales,"['00745048', '00745078', '00745091', '00745151..."
6,EnquetesEC,1958,1968,Enquetes et commentaires,1679,"['Office Radio Television France', 'Radio Tele...",Enquêtes et commentaires,"['PHD88012247', 'PHD94019542', 'PHD94025239', ..."
7,EuroDemain,1955,1955,L'Europe est pour demain,7,Radio Television Francaise,Construction européenne,"['PHD88014523', 'PHD88014524', 'PHD88014525', ..."
8,FChiffres,1958,1961,Des faits et des chiffres,150,Radio Television Francaise,Les banques suisses,"['PHD98203650', 'PHD98203650', 'PHD98203650', ..."
9,GASM,1969,1987,Les grandes avenues de la science moderne,733,"['Radio France', 'Office Radio Television Fran...",Energie nucléaire et armes nucléaires,"['PHD94024229', 'PHD94024230', 'PHD94025006', ..."


In [ ]:
aliases_df_filepath = f"{DATA_DIR}/data/sample_data/INA/collection_aliases.csv"
aliases_df = pd.read_csv(aliases_df_filepath, index_col=0)
aliases_df

,année_début,année_fin,Titre collection,Nombre de notices,Genres,Thématiques,Identifiants des notices
alias,,,,,,,
CAPVD,1975,1984,75... 2000 : Comprendre aujourd'hui pour vivre...,107,"['Débat', 'Lecture', 'Journal parlé', 'Magazine']","['Politique', 'Média', 'Education pédagogie', ...","['PHD95073186', 'PHD95073187', 'PHD95073188', ..."
CFCE,1996,1996,"C'est en France, c'est en Europe",17,"['Entretien', 'Reportage', 'Magazine']","['Politique', 'Tourisme', 'Société', 'Religion...","['00165368', '00167288', '00169364', '00171017..."
ConsEuro,1949,1960,Conseil de l'Europe,17,"['Déclaration', 'Chronique', 'Reportage', 'Jou...","['Politique', 'Information', 'Environnement']","['PHD86026894', 'PHD86026156', 'PHD86046313', ..."
EcoEH,1962,1968,L'économie et les hommes,295,"['Débat', 'Magazine', 'Causerie', 'Journal par...","['Politique', 'Beaux arts', 'Education pédagog...","['PHZ08001285', 'PHD94013850', 'PHD98010917', ..."
EdSpe,1958,1963,Edition spéciale,624,"['Bruitage image sonore', 'Magazine', 'Lecture...","['Education pédagogie', 'Cinéma', 'Tradition e...","['PHD98202592', 'PHD98203134', 'PHD98203149', ..."
EnjeuxInt,1984,1996,Les enjeux internationaux,1402,"['Magazine', 'Interview entretien']","['Politique', 'Média', 'Education pédagogie', ...","['PHD98025351', 'PHD98025624', 'PHD98026755', ..."
EnquetesEC,1958,1968,Enquêtes et commentaires,1679,"['Chronique', 'Magazine', 'Causerie', 'Journal...","['Education pédagogie', 'Cinéma', 'Tourisme', ...","['PHZ16058245', 'PHZ16058256', 'PHZ16058257', ..."
EuroDemain,1955,1955,L'Europe est pour demain,7,['Reportage'],['Information'],"['PHD88014525', 'PHD88014523', 'PHD88014524', ..."
FChiffres,1958,1961,Des faits et des chiffres,150,['Magazine'],"['Information', 'Industrie']","['PHD98207926', 'PHD98203650', 'PHD98203736', ..."


# 2. Create the issue index unifying all these views to be issue-based

Now we want to have a single file listing all the issues, with their respective date (if it's broadcast or recorded date), alias (thus collection title), the associated audio and text files. 
With this, we want a mirrored file which stored for each issue the associated metadata (collection, date, genre, theme, etc.) and the associated files (audio and text). 
These will be used together to detect/select the data, and access to issue-level metadata directly when importing the data into our canonical format.

In particular a few things will need to be checked specifically:
- The collection names are not exactly the same across the two files. The final file will need to have a clear and comprehensive view of what should be grouped. 
- All issues need a date, potentially approximated, either with the broadcast date or the recording date, but no issue should be without a date. entries `is_exact_data` and `is_broadcast_date` will be defined accordningly.
- The exact metada stored in the file will depend on what is available here, but adapt itself to the issue-level additional metadata we have for RTS and expect in the rebuilder, to ease the rebuilt process.

## a. Obtain the list of notices with corresponding files

First start from the list of notices and then the list of files.

 See if we have the same number of notices (issues in this case) and whether we can establish a match between each notice (issue) and the corresponding file(s).

In [8]:
# find if there are any duplicated notice IDs in the notice DF
duplicated_notice_ids = notice_df[notice_df['Identifiant de la notice'].duplicated()]
duplicated_notices = notice_df[notice_df.duplicated()]

msg = f"There are {len(notice_df)} notices in the DF, {len(duplicated_notices)} of which are duplicated notices ({len(duplicated_notice_ids)} dupl ids)"
print(msg)

duplicated_notices

There are 39607 notices in the DF, 16 of which are duplicated notices (16 dupl ids)


,Identifiant de la notice,Thèque,Titre collection,Titre propre,Date de diffusion,Date d'enregistrement,Heure de diffusion,Titre phonogramme,Genre,Thématique,Durée
4834,PHD98203846,PH (Phono),Paris vous parle,Les sports : boxe,09/01/1956,09/01/1956,19:15:00,NaN,Interview entretien ; Journal parlé ;,Sports ;,00:03:20
4835,PHD98203847,PH (Phono),Paris vous parle,CHABAN-DELMAS : la faiblesse de la nouvelle As...,09/01/1956,09/01/1956,19:15:00,NaN,Déclaration ; Journal parlé ;,Politique ;,00:00:55
4836,PHD98203854,PH (Phono),Paris vous parle,Rencontre franco espagnole sur l'avenir du Mar...,10/01/1956,10/01/1956,19:15:02,NaN,Journal parlé ; Papier ;,Politique ;,00:03:14
4837,PHD98203860,PH (Phono),Paris vous parle,Interview sur la nouvelle Assemblée nationale,10/01/1956,10/01/1956,19:15:00,NaN,Déclaration ; Interview entretien ; Journal pa...,Information ; Politique ;,00:00:53
6023,PHD98207103,PH (Phono),Paris vous parle,Enquête sur la santé mentale des Français,08/01/1956,08/01/1956,NaN,NaN,Interview entretien ; Journal parlé ;,Médecine santé ; Sociologie ;,NaN
6024,PHD98207104,PH (Phono),Paris vous parle,Enquête sur la santé mentale des Français,09/01/1956,09/01/1956,NaN,NaN,Interview entretien ; Journal parlé ;,Médecine santé ; Sociologie ;,NaN
6025,PHD98207105,PH (Phono),Paris vous parle,Enquête sur la santé mentale des Français,10/01/1956,23/12/1955,NaN,NaN,Interview entretien ; Journal parlé ;,Médecine santé ; Sociologie ;,NaN
6026,PHD98207163,PH (Phono),Paris vous parle,Suite enquête sur la santé mentale des Français,07/01/1956,07/01/1956,NaN,NaN,Interview entretien ; Journal parlé ;,Sociologie ;,NaN
6027,PHD98207166,PH (Phono),Paris vous parle,Paris vous parle du 6 janvier 1956,06/01/1956,06/01/1956,NaN,NaN,Déclaration ; Interview entretien ; Journal pa...,Information ;,NaN
6028,PHD98207167,PH (Phono),Paris vous parle,Arrivée en France de ministres marocains,06/01/1956,06/01/1956,NaN,NaN,Interview entretien ; Journal parlé ; Reportag...,Information ; Politique ;,00:02:00


There are some duplicated notice IDs, which also have the same information for all columns, so we can drop the duplicates.

In [9]:
# drop the duplicated notices
notice_df = notice_df.drop_duplicates()

print(len(notice_df))

39591


The notice ID dataframe does not actually have any information about the files, so we will need to match it with the file list, which has the notice ID as well.

In [11]:
print(len(ext_files_df))

47730


In [10]:
duplicated_files = ext_files_df[ext_files_df.duplicated()]
num_dups = len(duplicated_files)

files_df_no_dps = ext_files_df.drop_duplicates()

duplicated_notice_ids_in_files = files_df_no_dps[files_df_no_dps['Notice'].duplicated()]
len(duplicated_notice_ids_in_files), num_dups

(8122, 17)

There are 17 duplicated lines (removed) and 8122 duplicated notice IDs (with different file info), which correspond to Notices which rely on two different audio files.

### Trying to match the Notice Ids across the two files

In [ ]:
eg_id = "PHD97005479"
#eg_id = "PHF16035390"
eg_id_1 = "165368"
eg_id = "00165368"

# the values in column "Notice" are formatted in the same way as in notice_df['Identifiant de la notice']
eg_id_1 in ext_files_df["Identifiant notice"].values, eg_id in ext_files_df["Notice"].values, eg_id in notice_df['Identifiant de la notice'].values

(True, True, True)

In [12]:
# group the files df by Notice ID 
files_per_notice = files_df_no_dps.groupby('Notice').agg(set)
for col in files_per_notice.columns:
    files_per_notice[col] = files_per_notice[col].apply(lambda x: list(x)[0] if len(x)==1 else list(x))

files_per_notice = files_per_notice.reset_index()
files_per_notice

,Notice,Type de notice,Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion,Identifiant notice,Fichier source,Nom du fichier,TC IN,TC OUT,Format,Répertoire,Chemin,Titre propre,Titre collection,Chaine de diffusion,Heure de diffusion,Date de diffusion
0,00000090,EMISSION RAD.,Titre: Inter soir 19H00 du 1er janvier 1995 - ...,90,"[95F05001SA0001_01.MP3, 95F05001H1900SA0001_01...","[00000090_95F05001SA0001_01___EXPORT.MP3, 0000...","[:::, 00:00:00:00]","[:::, 00:00:00:00]",FVISIO,15- Inter soir ou Inter actualités 1990- fin j...,iMPRESSO\Journal_parle\15-_Inter_soir_ou_Inter...,Inter soir 19H00 du 1er janvier 1995,Inter soir 19h00,Radio France,19:00:00,01/01/1995
1,00000217,EMISSION RAD.,Titre: Inter soir 19H00 du 02 janvier 1995 - T...,217,"[95F05001H1900SA0002_01.MP3, 95F05001SA0002_01...",[00000217_95F05001H1900SA0002_01_1900_202200_E...,"[:::, 00:00:19:00]","[:::, 00:20:22:00]",FVISIO,15- Inter soir ou Inter actualités 1990- fin j...,iMPRESSO\Journal_parle\15-_Inter_soir_ou_Inter...,Inter soir 19H00 du 02 janvier 1995,Inter soir 19h00,Radio France,19:00:00,02/01/1995
2,00000218,EMISSION RAD.,Titre: EDUCATION : LACHEZ LEUR LES BASKETS ! -...,218,95F05001SA0002_01.MP3,00000218_95F05001SA0002_01___EXPORT.MP3,:::,:::,FVISIO,Le téléphone sonne,iMPRESSO\Magazines_d'information\Le_telephone_...,EDUCATION : LACHEZ LEUR LES BASKETS !,Le telephone sonne,Radio France,19:20:00,02/01/1995
3,00000345,EMISSION RAD.,Titre: Inter soir 19H00 du 03 janvier 1995 - T...,345,"[95F05001SA0003_01.MP3, 95F05001H1900SA0003_01...","[00000345_95F05001SA0003_01___EXPORT.MP3, 0000...",:::,:::,FVISIO,15- Inter soir ou Inter actualités 1990- fin j...,iMPRESSO\Journal_parle\15-_Inter_soir_ou_Inter...,Inter soir 19H00 du 03 janvier 1995,Inter soir 19h00,Radio France,19:00:00,03/01/1995
4,00000346,EMISSION RAD.,Titre: LE POUVOIR RUSSE MINE PAR LA CRISE DU C...,346,95F05001SA0003_01.MP3,00000346_95F05001SA0003_01___EXPORT.MP3,:::,:::,FVISIO,Le téléphone sonne,iMPRESSO\Magazines_d'information\Le_telephone_...,LE POUVOIR RUSSE MINE PAR LA CRISE DU CAUCASE,Le telephone sonne,Radio France,19:20:00,03/01/1995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39586,PHZ20001618,EMISSION RAD.,Titre: Chypre et l'OTAN - Titre collection:Par...,PHZ20001618,KB00928_01.MP3,PHZ20001618_KB00928_01_242110_254409_EXPORT.MP3,00:24:21:10,00:25:44:09,FVISIO,11- Paris vous parle 1958-1960,iMPRESSO\Journal_parle\11-_Paris_vous_parle_19...,Chypre et l'OTAN,Paris vous parle,Radio Television Francaise,19:36:21,23/09/1958
39587,PHZ20002367,EMISSION RAD.,Titre: Paris vous parle - Titre collection:Par...,PHZ20002367,LB18671_01.MP3,PHZ20002367_LB18671_01_0_0_EXPORT.MP3,00:00:00:00,00:00:00:00,FVISIO,11- Paris vous parle 1958-1960,iMPRESSO\Journal_parle\11-_Paris_vous_parle_19...,Paris vous parle,Paris vous parle,Radio Television Francaise,NaN,22/09/1958
39588,PHZ23000441,EMISSION RAD.,Titre: Inter actualites de 19H15 du 20 novembr...,PHZ23000441,KB18416_01.MP3,PHZ23000441_KB18416_01_0_0_EXPORT.MP3,00:00:00:00,00:00:00:00,FVISIO,12- Inter actualités 1960-1969,iMPRESSO\Journal_parle\12-_Inter_actualites_19...,Inter actualites de 19H15 du 20 novembre1961,Inter actualites de 19H15,Radio Television Francaise,19:14:00,20/11/1961
39589,PHZ23000447,EMISSION RAD.,Titre: Inter actualites de 19H15 du 11 decembr...,PHZ23000447,KB18917_01.MP3,PHZ23000447_KB18917_01_0_0_EXPORT.MP3,00:00:00:00,00:00:00:00,FVISIO,12- Inter actualités 1960-1969,iMPRESSO\Journal_parle\12-_Inter_actualites_19...,Inter actualites de 19H15 du 11 decembre 1961,Inter actualites de 19H15,Radio Television Francaise,19:14:59,11/12/1961


## b. Removing unecessary columns and reformating some columns

Columns to remove: 
- Type de notice
- Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion
- Format

Columns to reformat:
- Date de diffusion: extract the year, month and date. "Is broadcast date" will always be true. 
- Chemin and Nom du fichier: change '\\' into '/' combine to produce the actual paths to the files
- Titre collection: unify to ensure they can all be mapped to one of the pre-defined shows and corresponding alias. 
- TC IN and TC OUT: using some examples, check exactly what they correspond to and how they should be used to combine the multiple audio files for a given notice (issue).

*First step - The index*
Columns to add for the index:
- Alias: based on the unified collection name. 
- Local path: reformatted combination of "Chemin" and "Nom du fichier"
- xml filepath: should be the same as local path but with .xml extension instead of .mp3
- edition: based on the exact date, with the usual numbering function.

### 1. Removing unecessary columns

In [13]:
dropped_cols = [
    'Type de notice', 
    "Titre propre-Titre collection-date de diffusion-Chaine de diffusion-Heure de diffusion-Notice-Date de diffusion", 
    "Format"
]
files_per_notice = files_per_notice.drop(dropped_cols, axis=1)

### 2. Multi-audio notices with respect to TC IN and TC OUT

First we want to understand more the cases where we have multiple audios for the same notice, and how the TC IN and TC OUT look in these cases.
Create a df which only has more than one audio.

In [ ]:
multi_file_notice_df = files_per_notice[files_per_notice['Fichier source'].apply(lambda x: isinstance(x, list) and len(x)>1)]
len(multi_file_notice_df), any(multi_file_notice_df['Notice'].duplicated())

(5701, False)

In [35]:
# examples of elements taken at random
rand_idx = np.random.randint(len(multi_file_notice_df))
print(rand_idx)
multi_file_notice_df.iloc[rand_idx].values

1822


array(['00824393', '824393',
       list(['94F05001H1900SA0346_01.MP3', '94F05001SA0346_01.MP3']),
       list(['00824393_94F05001SA0346_01___EXPORT.MP3', '00824393_94F05001H1900SA0346_01___EXPORT.MP3']),
       ':::', ':::',
       '15- Inter soir ou Inter actualités 1990- fin juillet1997',
       'iMPRESSO\\Journal_parle\\15-_Inter_soir_ou_Inter_actualites_1990-_fin_juillet1997',
       'Inter soir 19H00 du 12 decembre 1994', 'Inter soir 19h00',
       'Radio France', '19:00:00', '12/12/1994'], dtype=object)

#### Report on the explored cases

1.  Notice id - PHD86043623 (idx = 2568) --> identical content, choose REEXT
    - Files - ['PHD86043623_DD18001_01_0_231207_EXPORT.MP3', 'PHD86043623_REEXT01020_01_0_191100_EXPORT.MP3']
    - Path - 'iMPRESSO\\Journal_parle\\08-Tribune_de_Paris_1946-1963'
    - Both audios actually contain exactly the same content, with a slight difference in length and timestamps. It's hard to know it will be the same file compared to other cases of multi-audio notices.
2.  Notice id - PHD86045727 (idx = 2854) --> identical content, choose REEXT
    - Files - ['PHD86045727_REEXT02274_01_0_192901_EXPORT.MP3', 'PHD86045727_DD32853_01___EXPORT.MP3']
    - Path - 'iMPRESSO\\Journal_parle\\08-Tribune_de_Paris_1946-1963'
    - Both audios actually contain exactly the same content, with a slight difference in length and timestamps. It's hard to know it will be the same file compared to other cases of multi-audio notices.
3.  Notice id - PHD98019184 (idx = 4019) --> unique issue with 2 audios
    - Files - ['PHD98019184_83C03044SA0019_01___EXPORT.MP3', 'PHD98019184_83C03044SA0019_02___EXPORT.MP3']
    - Path - 'iMPRESSO\\Magazines_d'information\\Le_monde_contemporain'
    - Now the second follows the first. there is _01 and _02 which could indicate that they are two parts of the same audio.
4.  Notice id - 00025683 (idx = 142) --> different issues
    - Files - ['00025683_95F05001H1900SA0117_01___EXPORT.MP3', '00025683_95F05001SA0117_01___EXPORT.MP3']
    - Path - 'iMPRESSO\\Journal_parle\\15-_Inter_soir_ou_Inter_actualites_1990-_fin_juillet1997'
    - Both audios seem to contain different news broadcasts, or at least different moments of the news broadcast, and don't necessarily seem to be closely related in terms of content or topic. They could probably be processed separately in different issues.
    "il est 8h" is present in "00025683_95F05001SA0117_01___EXPORT.MP3", and the other seems to take place at 7pm (and note H1900 can be found in its name, + it's mentioned during the first one).
5.  Notice id - PHD86045180 (idx = 2718) --> identical content, choose REEXT
    - Files - ['PHD86045180_REEXT01622_01_0_192614_EXPORT.MP3', 'PHD86045180_DD28635_01___EXPORT.MP3']
    - Path - 'iMPRESSO\\Journal_parle\\08-Tribune_de_Paris_1946-1963'
    - Same content again, but the filenames actually contain "REEXT", just like for the first two cases, which could indicate when the content will be the same AND allow us to know which one to keep.
6.  Notice id - PHD98205189 (idx = 4243) --> imbricated content
    - Files - ['PHD98205189_LB05784_01___EXPORT.MP3', 'PHD98205189_431D00187_01_200701_211501_EXPORT.MP3']
    - Path - 'iMPRESSO\\Journal_parle\\10-_Paris_vous_parle_1956-1957'
    - The first audio is completely included in the second audio, which contains lots of additional content. It's not directly clear what should be done for this exact case. Do we keep both? Only the second one? (First audio is very short but that does not seem like a viable criterion to identify such cases in the data).
7.  Notice id - 00613787 (idx = 1442) --> different issues
    - Files - ['00613787_94F05001H1900SA0097_01___EXPORT.MP3', '00613787_94F05001SA0097_01___EXPORT.MP3']
    - Path - 'iMPRESSO\\Magazines_d'information\\Le_telephone_sonne'
    - Again completely different shows, one at 7am the other at 7pm.
8.  Notice id - 00581783 (idx = 1318) --> imbricated or duplicated content
    - Files - ['00581783_94F05001SA0020_01_3263413_4065616_EXPORT.MP3', '00581783_94F05001H1900SA0020_01_0_0_EXPORT.MP3']
    - Path - 'iMPRESSO\\Magazines_d'information\\Le_telephone_sonne'
    - The contents of the two audios largerly overlaps, but one starts before and the other ends slightly after. From what I was able to see, the first audio is specifically focused on the "show" "le telephone sonne", while the other starts before the show start and ends slightly before. In this case there is no explicit way to differentiate them from the audio filename, except that the one which is focused on the show does not have "H1900" in its name, but it is not clear if this is a generalizable criterion.
9.  Notice id - PHD99204835 (idx = 4859) --> unique issue with 2 audios
    - Files - ['PHD99204835_186L00050_02___EXPORT.MP3', 'PHD99204835_186L00050_01___EXPORT.MP3']
    - Path - 'iMPRESSO\\Magazines_d'information\\Le_monde_contemporain'
    - One follows the other, like in example 3 and the order can be found in the filename.
9.  Notice id - PHD98208270 (idx = 4451) --> imbricated content
    - Files - ['PHD98208270_LB17259_01___EXPORT.MP3', 'PHD98208270_LO03273_01___EXPORT.MP3']
    - Path - 'iMPRESSO\\Journal_parle\\11-_Paris_vous_parle_1958-1960'
    - Again, one audio is completely included in the other, it's the case when it's a correspondant which can be heard, with a different recording compared to the main audio. This cannot be distinguished from the filenames. The imbricated audio is 14mins long which is too long to fit a possible "correspondant intervention" criterion.

List all the possible collection names to see if covered an honest amount of the collection.

The examples we found came mostly from:
- Tribune de Paris
- Le monde contemporain
- Inter actualites de 19H00 or inter soir de 19h00
- Le telephone sonne
- Paris vous parle

In [30]:
multi_file_notice_df['Titre collection'].value_counts()

Titre collection
Paris vous parle                                                              1179
Inter soir 19h00                                                              1174
Tribune de Paris : Les hommes  les evenements  les idees a l'ordre du jour     848
Le monde contemporain                                                          837
Le telephone sonne                                                             584
Universite radiophonique internationale                                        305
Rue des entrepreneurs                                                          204
Radio Actualites Francaises                                                    158
Inter actualites de 19H00                                                      131
Edition speciale                                                                46
Mode d'emploi de l'Europe                                                       27
Sciences et techniques                                                

In [36]:
multi_file_notice_df[multi_file_notice_df['Titre collection']=="Des faits et des chiffres"]

,Notice,Identifiant notice,Fichier source,Nom du fichier,TC IN,TC OUT,Répertoire,Chemin,Titre propre,Titre collection,Chaine de diffusion,Heure de diffusion,Date de diffusion
21161,PHD98203650,PHD98203650,"[78INA08505PH0135_04.MP3, 78INA08505PH0135_01....",[PHD98203650_78INA08505PH0135_04_255907_460318...,"[:::, 00:25:59:07]","[:::, 00:46:03:18]",Les banques suisses,iMPRESSO\Les_banques_suisses,Des faits et des chiffres : emission du 23 mai...,Des faits et des chiffres,Radio Television Francaise,18:30:00,23/05/1958
21168,PHD98203736,PHD98203736,"[80INA08505PH0035_02.MP3, LB16470_01.MP3, 80IN...","[PHD98203736_LB16470_01___EXPORT.MP3, PHD98203...","[:::, 02:08:52:07]","[:::, 02:38:49:00]",Les banques suisses,iMPRESSO\Les_banques_suisses,Des faits et des chiffres : emission du 13 jui...,Des faits et des chiffres,Radio Television Francaise,18:30:00,13/06/1958
21171,PHD98203792,PHD98203792,"[81INA08505PH0046_01.MP3, LB16695_01.MP3]",[PHD98203792_81INA08505PH0046_01_1091909_11419...,"[:::, 01:09:19:09]","[:::, 01:14:19:02]",Les banques suisses,iMPRESSO\Les_banques_suisses,La normalisation,Des faits et des chiffres,Radio Television Francaise,NaN,23/06/1958
21172,PHD98203816,PHD98203816,"[81INA08505PH0050_01.MP3, LB16786_01.MP3]",[PHD98203816_81INA08505PH0050_01_1362703_20313...,"[:::, 01:36:27:03]","[:::, 02:03:13:13]",Les banques suisses,iMPRESSO\Les_banques_suisses,Des faits et des chiffres : emission du 27 jui...,Des faits et des chiffres,Radio Television Francaise,18:30:00,27/06/1958


#### More than 2 audios

We now also have cases where there are more than two files associated to the same notice. This also needs to be investigated.

In [ ]:
# case #1 - 5 audios for this notice
multi_file_notice_df[multi_file_notice_df['Titre collection']=="Des faits et des chiffres"].iloc[0].values

array(['PHD98203650', 'PHD98203650',
       list(['78INA08505PH0135_04.MP3', '78INA08505PH0135_01.MP3', '78INA08505PH0135_02.MP3', 'LB15967_01.MP3', '78INA08505PH0135_03.MP3']),
       list(['PHD98203650_78INA08505PH0135_04_255907_460318_EXPORT.MP3', 'PHD98203650_78INA08505PH0135_01___EXPORT.MP3', 'PHD98203650_78INA08505PH0135_03___EXPORT.MP3', 'PHD98203650_LB15967_01___EXPORT.MP3', 'PHD98203650_78INA08505PH0135_02___EXPORT.MP3']),
       list([':::', '00:25:59:07']), list([':::', '00:46:03:18']),
       'Les banques suisses', 'iMPRESSO\\Les_banques_suisses',
       'Des faits et des chiffres : emission du 23 mai 1958',
       'Des faits et des chiffres', 'Radio Television Francaise',
       '18:30:00', '23/05/1958'], dtype=object)

It seems that the audio files which have the same prefix are following each other. The #4, which has a different suffix is a little harder to tell. Finally, PHD98203650_LB15967_01___EXPORT.MP3 which has a diffferent prefix seems to be a different issue altogether but it's hard to know for sure without listening to the entirety of the shows.

Should each audio be its own content-item??? Should it be one content-item spread amongst several files? That was the first idea but overs 3 audio files each spanning up to 1h it seems hard to do.

In [26]:
multi_file_notice_df.columns

Index(['Notice', 'Identifiant notice', 'Fichier source', 'Nom du fichier',
       'TC IN', 'TC OUT', 'Répertoire', 'Chemin', 'Titre propre',
       'Titre collection', 'Chaine de diffusion', 'Heure de diffusion',
       'Date de diffusion'],
      dtype='str')

In [59]:
"PHD99103312" in ext_files_df["Identifiant notice"].values, "27443" in ext_files_df["Identifiant notice"].values, "00027443" in ext_files_df["Notice"].values

(True, True, True)